<a href="https://colab.research.google.com/github/sreeakhilkolachina/Development-of-Interactive-Cyber-Threat-Visualization-Dashboard/blob/main/ai_generated_cyber_threats_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import sqlite3
import pandas as pd

# Create SQLite database
conn = sqlite3.connect("cyber_threat_detection.db")
cursor = conn.cursor()

# Create table
cursor.execute("""
CREATE TABLE IF NOT EXISTS cyber_threats (
    Threat_ID TEXT PRIMARY KEY,
    Timestamp TEXT,
    Source_IP TEXT,
    Destination_IP TEXT,
    Protocol TEXT,
    Port INTEGER,
    Packet_Size INTEGER,
    Login_Attempts INTEGER,
    Failed_Attempts INTEGER,
    Malware_Detected TEXT,
    Traffic_Type TEXT,
    Threat_Label INTEGER
)
""")

conn.commit()
print("Database and table created successfully!")


Database and table created successfully!


In [7]:
data = [
    ('T001','2023-01-10 10:15','192.168.1.10','10.0.0.5','TCP',80,450,2,0,'No','Normal',0),
    ('T002','2023-01-10 10:18','192.168.1.15','10.0.0.8','UDP',53,1200,1,0,'No','Normal',0),
    ('T003','2023-01-11 09:45','172.16.0.4','10.0.0.20','TCP',22,300,15,10,'No','Suspicious',1),
    ('T004','2023-01-11 10:05','172.16.0.9','10.0.0.25','TCP',443,800,3,1,'No','Normal',0),
    ('T005','2023-01-12 14:30','203.0.113.5','10.0.0.30','TCP',3389,1500,20,18,'No','Attack',1),
    ('T006','2023-01-12 15:00','203.0.113.8','10.0.0.35','TCP',445,2000,25,20,'Yes','Attack',1),
    ('T007','2023-01-13 11:20','192.168.2.5','10.0.0.40','UDP',161,400,2,0,'No','Normal',0),
    ('T008','2023-01-13 11:45','198.51.100.10','10.0.0.45','TCP',21,900,12,8,'No','Suspicious',1),
    ('T009','2023-01-14 16:10','198.51.100.20','10.0.0.50','TCP',25,1100,18,15,'Yes','Attack',1),
    ('T010','2023-01-15 09:00','192.168.3.3','10.0.0.55','TCP',80,500,1,0,'No','Normal',0)
]

cursor.executemany("""
INSERT OR REPLACE INTO cyber_threats VALUES (?,?,?,?,?,?,?,?,?,?,?,?)
""", data)

conn.commit()
print("Dummy data inserted successfully!")


Dummy data inserted successfully!


In [8]:
query = "SELECT * FROM cyber_threats"
pd.read_sql(query, conn)


,Threat_ID,Timestamp,Source_IP,Destination_IP,Protocol,Port,Packet_Size,Login_Attempts,Failed_Attempts,Malware_Detected,Traffic_Type,Threat_Label
0,T001,2023-01-10 10:15,192.168.1.10,10.0.0.5,TCP,80,450,2,0,No,Normal,0
1,T002,2023-01-10 10:18,192.168.1.15,10.0.0.8,UDP,53,1200,1,0,No,Normal,0
2,T003,2023-01-11 09:45,172.16.0.4,10.0.0.20,TCP,22,300,15,10,No,Suspicious,1
3,T004,2023-01-11 10:05,172.16.0.9,10.0.0.25,TCP,443,800,3,1,No,Normal,0
4,T005,2023-01-12 14:30,203.0.113.5,10.0.0.30,TCP,3389,1500,20,18,No,Attack,1
5,T006,2023-01-12 15:00,203.0.113.8,10.0.0.35,TCP,445,2000,25,20,Yes,Attack,1
6,T007,2023-01-13 11:20,192.168.2.5,10.0.0.40,UDP,161,400,2,0,No,Normal,0
7,T008,2023-01-13 11:45,198.51.100.10,10.0.0.45,TCP,21,900,12,8,No,Suspicious,1
8,T009,2023-01-14 16:10,198.51.100.20,10.0.0.50,TCP,25,1100,18,15,Yes,Attack,1
9,T010,2023-01-15 09:00,192.168.3.3,10.0.0.55,TCP,80,500,1,0,No,Normal,0


In [9]:
query = """
SELECT COUNT(*) AS Total_Threats
FROM cyber_threats
WHERE Threat_Label = 1
"""
pd.read_sql(query, conn)


,Total_Threats
0,5


In [10]:
query = """
SELECT Traffic_Type, COUNT(*) AS Count
FROM cyber_threats
GROUP BY Traffic_Type
"""
pd.read_sql(query, conn)


,Traffic_Type,Count
0,Attack,3
1,Normal,5
2,Suspicious,2


In [11]:
query = """
SELECT Source_IP, SUM(Failed_Attempts) AS Total_Failed_Attempts
FROM cyber_threats
GROUP BY Source_IP
HAVING Total_Failed_Attempts > 5
"""
pd.read_sql(query, conn)


,Source_IP,Total_Failed_Attempts
0,172.16.0.4,10
1,198.51.100.10,8
2,198.51.100.20,15
3,203.0.113.5,18
4,203.0.113.8,20


In [12]:
query = """
SELECT *
FROM cyber_threats
WHERE Malware_Detected = 'Yes'
"""
pd.read_sql(query, conn)


,Threat_ID,Timestamp,Source_IP,Destination_IP,Protocol,Port,Packet_Size,Login_Attempts,Failed_Attempts,Malware_Detected,Traffic_Type,Threat_Label
0,T006,2023-01-12 15:00,203.0.113.8,10.0.0.35,TCP,445,2000,25,20,Yes,Attack,1
1,T009,2023-01-14 16:10,198.51.100.20,10.0.0.50,TCP,25,1100,18,15,Yes,Attack,1


In [13]:
query = """
SELECT Port, COUNT(*) AS Attack_Count
FROM cyber_threats
WHERE Threat_Label = 1
GROUP BY Port
ORDER BY Attack_Count DESC
"""
pd.read_sql(query, conn)


,Port,Attack_Count
0,3389,1
1,445,1
2,25,1
3,22,1
4,21,1


In [14]:
query = """
SELECT substr(Timestamp,1,10) AS Date, COUNT(*) AS Attacks
FROM cyber_threats
WHERE Threat_Label = 1
GROUP BY Date
"""
pd.read_sql(query, conn)


,Date,Attacks
0,2023-01-11,1
1,2023-01-12,2
2,2023-01-13,1
3,2023-01-14,1
